# 04B - Tabular-only NN ablation

Diagnostic side-notebook, not part of the 00->06 report sequence. Isolates how much the 2000-column TF-IDF block is buying the PyTorch NN (notebook 04) versus the 44 engineered tabular columns alone. Same `WineScorePredictorNet`/`NNTuner`/`NNTuningConfig` pipeline and the same `configs/nn_training.json` values, pointed at a 44-column `X` instead of the concatenated 2044-column one.

## Brief

Baseline + full Optuna search (same config as notebook 04) on tabular-only features, evaluated with the same metrics, persisted to separate files so notebook 04's artifacts are untouched.

In [1]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import optuna
import torch
import wandb
from dotenv import load_dotenv
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

from diplo_mod_1.constants import CONFIGS, MODELS, PROCESSED, RANDOM_STATE, REPORTS
from diplo_mod_1.schemas.evaluation import evaluate_predictor
from diplo_mod_1.training import (
    NNTuner,
    NNTuningConfig,
    WineScorePredictorNet,
    detect_torch_device,
)
from diplo_mod_1.training.config import RunRecord, TuningHistory

# override=True so re-running this cell after editing .env (without a kernel
# restart) always picks up the new values, instead of keeping whatever was
# already loaded into os.environ earlier in this kernel session.
load_dotenv(override=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Experiment tracking is opt-in: set WANDB_ENABLED=true (in .env or the shell)
# to log this run to Weights & Biases. Off by default so `poe check` / nbmake
# executions don't create a W&B run on every lint/test pass.
WANDB_ENABLED = os.environ.get("WANDB_ENABLED", "false").lower() == "true"

## Step 1 - Load tabular-only dataset (44 columns, no TF-IDF)

In [2]:
nn_dir = PROCESSED / "nn"
X = {s: np.load(nn_dir / f"X_tab_{s}.npy") for s in ("train", "val", "test")}
y = {s: np.load(nn_dir / f"y_{s}.npy") for s in ("train", "val", "test")}

feature_meta = json.loads((nn_dir / "feature_names.json").read_text(encoding="utf-8"))
feature_names = feature_meta["feature_names"]

print(f"Tabular-only feature count: {len(feature_names)}")
print(f"X_train shape: {X['train'].shape}")

Tabular-only feature count: 44
X_train shape: (83180, 44)


## Step 2 - Baseline model

In [3]:
nn_config_name = os.environ.get("NN_TRAINING_CONFIG", "nn_training.json")
nn_config = NNTuningConfig.from_json(CONFIGS / nn_config_name)
print(f"Loaded NN config: {nn_config_name}")

run_id = f"nn_tabular_ablation-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

if WANDB_ENABLED:
    wandb.init(
        project=os.environ.get("WANDB_PROJECT", "diplo-mod-1"),
        name=f"nn-{run_id}",
        group="nn-tabular-ablation",
        job_type="hpo",
        config={
            "nn_config_name": nn_config_name,
            "feature_count": X["train"].shape[1],
            **nn_config.model_dump(),
        },
    )

baseline = WineScorePredictorNet(
    input_dim=X["train"].shape[1],
    hidden_sizes=[128, 64],
    max_epochs=nn_config.max_epochs,
    early_stopping_patience=nn_config.early_stopping_patience,
    random_state=RANDOM_STATE,
    device=detect_torch_device(),
)
baseline.fit(X["train"], y["train"], X_val=X["val"], y_val=y["val"])

baseline_pred = baseline.predict(X["val"])
baseline_rmse = root_mean_squared_error(y["val"], baseline_pred)
baseline_mae = mean_absolute_error(y["val"], baseline_pred)
baseline_r2 = r2_score(y["val"], baseline_pred)
print(f"Baseline val RMSE: {baseline_rmse:.4f}")
print(f"Baseline val MAE:  {baseline_mae:.4f}")
print(f"Baseline val R2:   {baseline_r2:.4f}")

if WANDB_ENABLED:
    wandb.log(
        {
            "baseline_val_rmse": baseline_rmse,
            "baseline_val_mae": baseline_mae,
            "baseline_val_r2": baseline_r2,
        }
    )

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


Loaded NN config: nn_training.json


wandb: Currently logged in as: leonardo-heis (leonardo-a-heis) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


PyTorch device: cuda
Baseline val RMSE: 2.1456
Baseline val MAE:  1.6846
Baseline val R2:   0.5028


## Step 3 - Hyperparameter tuning (Optuna)

Same `configs/nn_training.json` as notebook 04, unchanged.

In [4]:
def log_trial_to_wandb(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    wandb.log({"trial": trial.number, "val_rmse": trial.value, **trial.params})


tuner = NNTuner(nn_config)
callbacks = [log_trial_to_wandb] if WANDB_ENABLED else None
study = tuner.tune(X["train"], y["train"], X["val"], y["val"], callbacks=callbacks)

print(f"Best val RMSE: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

if WANDB_ENABLED:
    wandb.log(
        {
            "best_val_rmse": study.best_value,
            **{f"best_{k}": v for k, v in study.best_params.items()},
        }
    )

  0%|          | 0/10 [00:00<?, ?it/s]

Best val RMSE: 1.9474
Best params: {'architecture': '512_128_32', 'dropout': 0.15636968998990508, 'learning_rate': 0.0040215545266902904, 'weight_decay': 1.987021538542864e-06, 'batch_size': 64}


## Step 4 - Final model + per-epoch W&B logging

In [5]:
def log_epoch_to_wandb(epoch: int, train_loss: float, val_loss: float) -> None:
    wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})


epoch_callbacks = [log_epoch_to_wandb] if WANDB_ENABLED else None
best_model = tuner.fit_best(
    X["train"], y["train"], X["val"], y["val"], study, callbacks=epoch_callbacks
)
print(f"Best epoch: {best_model.best_epoch_} / {len(best_model.train_losses_)} trained")

Best epoch: 8 / 14 trained


## Step 5 - Evaluate on train / val / test

Same `evaluate_predictor` call and metrics shape as notebook 04.

In [6]:
splits = {
    "train": (X["train"], y["train"]),
    "val": (X["val"], y["val"]),
    "test": (X["test"], y["test"]),
}
result = evaluate_predictor(best_model, splits, model_type="neural_net")

for m in result.metrics:
    print(f"{m.split:5s}  RMSE={m.rmse:.4f}  MAE={m.mae:.4f}  R2={m.r2:.4f}")
    if WANDB_ENABLED:
        wandb.log({f"{m.split}_rmse": m.rmse, f"{m.split}_mae": m.mae, f"{m.split}_r2": m.r2})

train  RMSE=1.6776  MAE=1.3004  R2=0.6950
val    RMSE=1.9474  MAE=1.5224  R2=0.5903
test   RMSE=1.9576  MAE=1.5291  R2=0.5863


## Step 6 - Persist model and metrics (separate files, not via `NNModelRegistry`)

`NNModelRegistry.save_run` unconditionally overwrites `models/nn_best.pt` whenever its own history's best run changes - with a single ablation run in its own history, that's every time, which would clobber the real NN's checkpoint that notebook 05 uses. A subdirectory doesn't work either: `.gitignore`'s `models/*` + `!models/*.pt` negation only re-includes files directly under `models/`, not nested subdirectories. So this persists manually, reusing the same `RunRecord`/`TuningHistory` schema, to separate top-level files instead.

In [7]:
model_filename = "nn_tabular_ablation.pt"
torch.save(
    {
        "state_dict": best_model.model_.state_dict(),
        "input_dim": best_model.input_dim,
        "hidden_sizes": best_model.hidden_sizes,
        "dropout": best_model.dropout,
    },
    MODELS / model_filename,
)
record = RunRecord(
    run_id=run_id,
    tuning_config=nn_config_name,
    model_filename=model_filename,
    best_params=study.best_params,
    metrics=result.metrics,
)
history = TuningHistory(runs=[record], best_run_id=run_id)
(REPORTS / "nn_tabular_ablation_metrics.json").write_text(
    history.model_dump_json(indent=2), encoding="utf-8"
)
print(f"Model saved to {MODELS / model_filename}")
print(f"Metrics saved to {REPORTS / 'nn_tabular_ablation_metrics.json'}")

if WANDB_ENABLED:
    artifact = wandb.Artifact("nn_tabular_ablation_model", type="model")
    artifact.add_file(str(MODELS / model_filename))
    wandb.log_artifact(artifact)
    wandb.finish()

Model saved to ..\models\nn_tabular_ablation.pt
Metrics saved to ..\reports\nn_tabular_ablation_metrics.json


baseline_val_mae,▁
baseline_val_r2,▁
baseline_val_rmse,▁
batch_size,▃█▃▃▃▃█▁█▃
best_batch_size,▁
best_dropout,▁
best_learning_rate,▁
best_val_rmse,▁
best_weight_decay,▁
dropout,▆█▁▆▄▅▂▁▇▃
+15,...


## Step 7 - Compare against the full-feature NN (notebook 04)

In [8]:
metrics_path = REPORTS / "nn_metrics.json"
if metrics_path.exists():
    full_history = TuningHistory.model_validate_json(metrics_path.read_text(encoding="utf-8"))
    full_best = next(r for r in full_history.runs if r.run_id == full_history.best_run_id)
    full_test = next(m for m in full_best.metrics if m.split == "test")
    ablation_test = next(m for m in result.metrics if m.split == "test")
    print(f"Full NN   (2044 cols) test RMSE={full_test.rmse:.4f}  R2={full_test.r2:.4f}")
    print(f"Tabular NN  (44 cols) test RMSE={ablation_test.rmse:.4f}  R2={ablation_test.r2:.4f}")
else:
    print("reports/nn_metrics.json not found yet - run notebook 04 first for a full comparison.")

Full NN   (2044 cols) test RMSE=1.5950  R2=0.7254
Tabular NN  (44 cols) test RMSE=1.9576  R2=0.5863


## Notes

This notebook isolates the tabular-only signal (44 engineered columns, no TF-IDF) using the exact same tuning budget and pipeline as notebook 04, to answer whether the 2000-column TF-IDF block is worth its cost for the NN. Results here inform the next iteration of notebook 04 (wider architecture search, more trials, LR scheduler, and/or revisiting the single flat-concatenation design if the TF-IDF block turns out to add little) rather than replacing it — notebook 04 stays untouched as the candidate for the report's model comparison.